# 03 — Experiment with model code directly

Run model code **inside this notebook's Python kernel**. Choose the model and
use `csv_input` from the shared `notebooks/config.json`, just as notebook 02 does.
The input may be an S3 URI or a local CSV path. Notebook 01 is not required when
the CSV already exists.

Read the model's existing parameters without editing them. Inspect a small
dataframe, then run selection → preparation → training → prediction →
post-processing. The notebook sets its environment variables explicitly before
importing model code. Restart the kernel when changing models or code that
loads settings at import time.

This notebook does not launch SageMaker jobs, register models or install
packages. Use the model's supported Python environment. Its local training
sample is for experimentation, not a held-out accuracy measurement.

For **post-processing only**, jump to **Create the SageMaker post-processing
folders** below. That section works in a fresh kernel with files from an existing
inference run. You can edit and rerun post-processing without training or scoring.


## Choose the model and read the shared config

- Locate the checkout and load notebooks/config.json.
- Choose the model and use the existing csv_input from that config.
- Create a fresh local output folder for this experiment.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "mlops-models/src/mlops_models").is_dir())
sys.path[:0] = [str(ROOT / "mlops-models/src"), str(ROOT / "mlops-models")]

config_path = ROOT / "notebooks/config.json"
notebook_config = json.loads(config_path.read_text())
csv_input = notebook_config.get("csv_input", "").strip()
if not csv_input:
    raise ValueError(f"Set csv_input in {config_path} to the shared S3 URI or a local CSV.")

MODEL_ID = "usecase1"       # Choose your model folder.
TRAINING_MODE = "1"        # Preliminary training; your trainer owns the mode behavior.
MAX_ROWS = 200             # Keep the exploratory sample small.

MODEL_SOURCE = ROOT / "mlops-models/src/mlops_models/models" / MODEL_ID
RUN = ROOT / ".runs/model-experiments" / MODEL_ID / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f")
RUN.mkdir(parents=True)
print("Kernel:", sys.executable)
print("Model source:", MODEL_SOURCE)
print("CSV:", csv_input)
print("Local outputs:", RUN)


## Set the notebook environment

- Set the model identity, training mode and CSV analysis mode before imports.
- Point model/output variables at local folders; the parameter folder is the model source until training finishes.
- Disable managed tracking and run reports for this local experiment.


In [ ]:
model_directory = RUN / "model"
training_reports = RUN / "training-reports"
model_directory.mkdir(exist_ok=True)
training_reports.mkdir(exist_ok=True)

# These values apply to this notebook kernel. No dotenv file is needed.
notebook_environment = {
    "MODEL_ID": MODEL_ID,
    "MLOPS_TRAINING_MODE": TRAINING_MODE,
    "MLOPS_MODEL_DIR": str(MODEL_SOURCE),  # Existing parameters, before a saved model is available.
    "SM_MODEL_DIR": str(model_directory),
    "SM_OUTPUT_DATA_DIR": str(training_reports),
    "SM_OUTPUT_DIR": str(training_reports),
    "MLOPS_ANALYSIS_MODE": "prediction_inference",
    "MLOPS_PREDICTION_FORMAT": "csv",
    "MLOPS_MODEL_METADATA_URI": "none",   # Preliminary training starts without prior metadata.
    "MLOPS_TRACKING_URI": "",
    "MLOPS_WRITE_RUN_REPORT": "0",
    "MLOPS_LOG_ARTIFACTS": "0",
    "TUNING_OBJECTIVE": "",
}
os.environ.update(notebook_environment)
for name, value in notebook_environment.items():
    print(name, "=", repr(value))


## Read the model parameters

- Read the selected model's existing parameters.json.
- Keep those settings unchanged and display them for reference.
- Start preliminary training without prior model metadata.


In [ ]:
import pandas as pd
from IPython.display import display

parameter_path = MODEL_SOURCE / "parameters.json"
parameters = json.loads(parameter_path.read_text())
metadata_input = {}
print("Parameters:", parameter_path)
display(parameters)


## Inspect a small dataframe

- Read the configured CSV with the notebook's AWS identity when it is on S3.
- Keep the ID column as text and load at most MAX_ROWS.
- Display the dataframe; this cell does not transform or save it.


In [ ]:
read_options = dict(nrows=MAX_ROWS, dtype={parameters["primary_key"]: str},
                    keep_default_na=False, na_values=["", r"\N"])
if csv_input.startswith("s3://"):
    import boto3
    bucket, key = csv_input.removeprefix("s3://").split("/", 1)
    with boto3.client("s3").get_object(Bucket=bucket, Key=key)["Body"] as stream:
        df = pd.read_csv(stream, **read_options)
else:
    csv_path = Path(csv_input).expanduser()
    if not csv_path.is_absolute():
        csv_path = ROOT / csv_path
    df = pd.read_csv(csv_path, **read_options)
print("CSV:", csv_input, "| rows / columns:", df.shape)
display(df.head())


## Import the code you want to experiment with

- Import the real base selector, preparation and training modules.
- Show their source files and the framework class behind ModelTraining.
- Keep these imports editable when testing a use-case override.

For example, import your use case's selector or trainer instead of the base
class when testing an override. After editing source, restart the kernel and
run from the top for a clean experiment. In VS Code, set a breakpoint in the
imported `.py` file and use **Debug Cell** to step into the actual function.

In [ ]:
from mlops_models.model_base.training import feature_selection, preprocess, train

for module in (feature_selection, preprocess, train):
    print(module.__name__, "->", module.__file__)
print("Training class:", train.ModelTraining.__module__, train.ModelTraining.__name__)

## Fit and inspect the selector

- Construct the real selector with the experiment settings.
- Fit it to the dataframe and retain the selector object for inspection.
- Export its selected-feature CSV and PDF through the existing output function.

In [ ]:
selector = feature_selection.FeatureSelector(parameters)
selector.fit(df)
selection_directory = feature_selection.write_container_output(selector, RUN)
selection_file = selection_directory / "selected_features.csv"
display(pd.read_csv(selection_file))
print("Selection outputs:", selection_directory)

## Run the model's training preparation

- Save the inspected dataframe locally because the preparation function accepts a directory containing data.csv.
- Pass that directory, selected features and the original parameter path to the model code.
- Inspect the prepared dataframe; the shared source CSV stays unchanged.


In [ ]:
raw_directory = RUN / "raw"
raw_directory.mkdir(exist_ok=True)
df.to_csv(raw_directory / "data.csv", index=False)
preprocess.process_train(raw_directory, selection_file, RUN, metadata_input, parameter_path)
prepared_directory = RUN / "prepared"
prepared = pd.read_csv(prepared_directory / "data.csv", dtype={parameters["primary_key"]: str})
print("Prepared shape:", prepared.shape)
display(prepared.head())


## Train directly and retain the model outputs

- Construct the trainer with the original settings and the model identity supplied by the application.
- Train using the local feature and data paths; keep the trainer and metadata visible.
- Let the existing packaging function save the model and its settings for inference.

This cell performs real training in the notebook environment. Run from the top
for a fresh experiment. Evaluation on this sample is not an independent test.


In [ ]:
# Supply the runtime model identity without changing parameters or its source file.
trainer = train.ModelTraining(
    selection_file, prepared_directory, metadata_input, {**parameters, "model_id": MODEL_ID})
checkpoint, model_metadata = trainer.train_model(RUN / "checkpoint", training_reports)
model_directory = train.write_container_output(
    checkpoint, model_metadata, model_directory, training_reports, parameters=parameters)
print("Saved model:", model_directory)
display(model_metadata)


## Set up prediction using the saved model

- Point the environment at this experiment's saved model and score output folder before inference imports.
- Read the parameters and metadata saved with that model.
- Run inference preparation on the local snapshot with the saved selection.


In [ ]:
score_directory = RUN / "scores"
score_directory.mkdir(exist_ok=True)
os.environ["MLOPS_MODEL_DIR"] = str(model_directory.resolve())
os.environ["SM_MODEL_DIR"] = str(model_directory.resolve())
os.environ["SM_OUTPUT_DATA_DIR"] = str(score_directory)
os.environ["SM_OUTPUT_DIR"] = str(score_directory)
saved_parameters = json.loads((model_directory / "parameters.json").read_text())
model_metadata = json.loads((model_directory / "model_metadata.json").read_text())
from mlops_models.model_base.inference import preprocess as inference_preprocess
from mlops_models.model_base.inference import predict

inference_preprocess.preprocess_data(
    raw_directory, selection_file, RUN / "inference", model_metadata,
    model_directory / "parameters.json")
inference_directory = RUN / "inference/prepared"
inference_frame = pd.read_csv(inference_directory / "data.csv",
                              dtype={saved_parameters["primary_key"]: str})
print("Model directory:", os.environ["MLOPS_MODEL_DIR"])
display(inference_frame.head())


## Inspect predictions and write the model's files

- Load the saved checkpoint through `gluonPredictor`, imported by the base as `ModelPredictor`.
- Keep a small prediction dataframe visible for experiments.
- Let the predictor write its normal CSV outputs into this experiment's score folder.

In [ ]:
predictor = predict.ModelPredictor(model_directory, model_metadata, saved_parameters)
print("Predictor class:", type(predictor).__module__, type(predictor).__name__)
predictions = predictor.predict_with_best_model(inference_frame.head(10))
display(predictions)
predictor.predict_file(
    inference_directory / "data.csv", model_metadata["feature_mapping"],
    saved_parameters["primary_key"], output_directory=score_directory)
predict.write_container_output(score_directory)


## Create the SageMaker post-processing folders

- Create the same input folders used by this application's SageMaker Processing step.
- Set the model and output environment before importing post-processing code.
- Print the paths so you can inspect or populate them directly.

**Start here for post-processing only.** The cells below do not need the earlier
training cells. Keep the default `/opt/ml` to test absolute path references.
If your notebook user cannot create these folders, have the workspace owner make
`/opt/ml/processing` writable; this notebook does not change permissions. You can
explicitly change ML_ROOT for tests that use the supplied paths, but that does
not test hard-coded `/opt/ml` references.

Use inputs from the same inference execution and its selected model:

| Input | Location beneath `/opt/ml/processing` |
| --- | --- |
| Original data snapshot (before preprocessing) | `raw/data.csv` |
| Selected features used by that model | `selection/selected_features.csv` |
| Prediction CSV files, retaining their subfolders | `input/` |
| Unpacked model, including parameters.json and model_metadata.json | `model/` |
| Files written by post-processing | `output/` |

These are local folders in your notebook environment. No SageMaker job starts.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "mlops-models/src/mlops_models").is_dir())
sys.path[:0] = [str(ROOT / "mlops-models/src"), str(ROOT / "mlops-models")]

POSTPROCESS_MODEL_ID = globals().get("MODEL_ID", "usecase1")  # Choose the model for these files.
ML_ROOT = Path("/opt/ml")
ANALYSIS_MODE = "prediction_inference"  # Or "prediction_test" for labelled test data.

postprocess_root = ML_ROOT / "processing"
postprocess_data = postprocess_root / "raw"
postprocess_selection = postprocess_root / "selection/selected_features.csv"
postprocess_predictions = postprocess_root / "input"
postprocess_model = postprocess_root / "model"
analysis_directory = postprocess_root / "output"
for folder in (postprocess_data, postprocess_selection.parent, postprocess_predictions,
               postprocess_model, analysis_directory):
    folder.mkdir(parents=True, exist_ok=True)

os.environ.update({
    "MODEL_ID": POSTPROCESS_MODEL_ID,
    "MLOPS_MODEL_DIR": str(postprocess_model),
    "SM_MODEL_DIR": str(postprocess_model),
    "SM_OUTPUT_DATA_DIR": str(analysis_directory),
    "SM_OUTPUT_DIR": str(analysis_directory),
    "MLOPS_ANALYSIS_MODE": ANALYSIS_MODE,
    "MLOPS_PREDICTION_FORMAT": "csv",
    "MLOPS_TRACKING_URI": "",
    "MLOPS_WRITE_RUN_REPORT": "0",
    "MLOPS_LOG_ARTIFACTS": "0",
})
for name, value in (("data", postprocess_data / "data.csv"),
                    ("selection", postprocess_selection), ("predictions", postprocess_predictions),
                    ("model", postprocess_model), ("output", analysis_directory)):
    print(name, "->", value)


## Put existing inputs into those folders

- Use outputs already produced above, or replace the four source paths with files downloaded from an inference job.
- Copy inputs unchanged, retaining prediction subfolders; a model.tar.gz is unpacked with Python's data extraction filter.
- Require empty destination input folders so inputs from different runs are not mixed.

When starting here in a fresh kernel, all four source values are initially None.
Fill them in to copy local files, or leave all four as None and place the files
in the printed folders yourself. The model source can be a directory or its
`.tar.gz` archive. This cell does not download or regenerate inputs.

Copy once. While editing post-processing, rerun only the code and output cells
below. To switch input sets, move aside the existing input folders first; this
cell does not delete or overwrite an existing set.


In [ ]:
import shutil
import tarfile

RAW_INPUT = globals().get("raw_directory")           # Directory containing data.csv, or that CSV file.
SELECTION_INPUT = globals().get("selection_file")    # selected_features.csv file.
PREDICTION_INPUT = globals().get("score_directory")   # Directory containing prediction CSV files.
MODEL_INPUT = globals().get("model_directory")       # Unpacked model directory or model.tar.gz.

sources = (RAW_INPUT, SELECTION_INPUT, PREDICTION_INPUT, MODEL_INPUT)
if all(value is None for value in sources):
    print("Place the inputs in the folders printed above, then run post-processing below.")
else:
    if any(value is None for value in sources):
        raise ValueError("Supply all four source paths, or leave all four as None to place files yourself.")
    raw_source, selection_source, prediction_source, model_source = map(Path, sources)
    if raw_source.is_dir():
        raw_source = raw_source / "data.csv"
    for item in (raw_source, selection_source):
        if not item.is_file():
            raise FileNotFoundError(item)
    if not prediction_source.is_dir() or not model_source.exists():
        raise FileNotFoundError("Check PREDICTION_INPUT and MODEL_INPUT.")
    for folder in (postprocess_data, postprocess_selection.parent,
                   postprocess_predictions, postprocess_model):
        if any(folder.iterdir()):
            raise FileExistsError(f"{folder} already has inputs. Reuse them by running post-processing below.")
    shutil.copy2(raw_source, postprocess_data / "data.csv")
    shutil.copy2(selection_source, postprocess_selection)
    shutil.copytree(prediction_source, postprocess_predictions, dirs_exist_ok=True)
    if model_source.is_dir():
        shutil.copytree(model_source, postprocess_model, dirs_exist_ok=True)
    else:
        with tarfile.open(model_source) as archive:
            archive.extractall(postprocess_model, filter="data")
    print("Inputs copied to:", postprocess_root)


## Edit and run inference post-processing locally

- Check the required input files, then import the base post-processing module from this checkout.
- Reload that module on subsequent runs so edits and module-level environment reads take effect.
- Call the real post-processing function with the SageMaker paths; inspect its variables or step into its functions.

Edit the `.py` file printed below, then rerun this cell. In VS Code, place a
breakpoint in that file and use **Debug Cell**. You can also call individual
functions on `postprocess` from another cell. Change module_name to your use-case
module if that is what you are editing. Restart the kernel and rerun this section
when changing imported dependencies or switching models.

This runs your local checkout's code. It does not load code from the model archive.
Your post-processing code owns the output files; reruns use the same output folder.


In [ ]:
for required in (postprocess_data / "data.csv", postprocess_selection,
                 postprocess_model / "parameters.json", postprocess_model / "model_metadata.json"):
    if not required.is_file():
        raise FileNotFoundError(f"Place the matching inference input here: {required}")
if not any(p.is_file() and p.suffix.lower() == ".csv" for p in postprocess_predictions.rglob("*")):
    raise FileNotFoundError(f"Place the prediction CSV files under {postprocess_predictions}")

module_name = "mlops_models.model_base.inference.postprocess"
if module_name in sys.modules:
    postprocess = importlib.reload(sys.modules[module_name])
else:
    postprocess = importlib.import_module(module_name)
print("Post-processing code:", postprocess.__file__)
postprocess.postprocess_predictions(
    postprocess_data, postprocess_selection, postprocess_predictions,
    analysis_directory, postprocess_model,
    analysis_mode=os.environ["MLOPS_ANALYSIS_MODE"])


## Inspect the local analysis outputs

- List files written beneath the configured output directory.
- Preview prediction_analysis.csv if your post-processing creates that file.
- Keep the other reports and plots available for inspection in the file explorer.


In [ ]:
for output_file in sorted(analysis_directory.rglob("*")):
    if output_file.is_file():
        print(output_file)
analysis_csv = analysis_directory / "prediction_analysis.csv"
if analysis_csv.is_file():
    import pandas as pd
    from IPython.display import display
    analysis_parameters = json.loads((postprocess_model / "parameters.json").read_text())
    display(pd.read_csv(analysis_csv, dtype={analysis_parameters["primary_key"]: str}).head())


## Continue the experiment

Inspect the intermediate objects from the cells you ran, or call individual
functions from the `postprocess` module. Keep the model's algorithms and file writers.

The earlier training workflow creates a fresh folder under `.runs` each time.
The post-processing section deliberately uses `/opt/ml/processing` so absolute
SageMaker path references are exercised. Its inputs stay in place for repeated
local debugging. Use **02** to publish reviewed code, then **04** to start or
inspect managed training/inference.
